In [0]:
%pip install pymysql sqlalchemy

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
#Your Railway MYSQL credentials
MYSQL_HOST = "caboose.proxy.rlwy.net"
MYSQL_PORT = 35868
MYSQL_DATABASE = "railway"
MYSQL_USER = "root"
MYSQL_PASSWORD ="kVJZkCqYbQTXHpDAVuOzYIByWLAsRaEa"

#Test connection
import pymysql
connection = pymysql.connect(
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    user=MYSQL_USER,
    password=MYSQL_PASSWORD,
    database=MYSQL_DATABASE
)

print("- Connected to MYSQL successfully!")
connection.close()


- Connected to MYSQL successfully!


In [0]:
#Create tables in MySQL
Cursor = connection = pymysql.connect(
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    user=MYSQL_USER,
    password=MYSQL_PASSWORD,
    database=MYSQL_DATABASE
)
cursor = connection.cursor()



#Create route_kpis table
cursor.execute("Drop Table if exists route_kpis")
cursor.execute("""
               create table if not exists route_kpis(
                   Origin VARCHAR(255),
                   Dest VARCHAR(255),
                   Route VARCHAR(255),
                   Month INT,
                   TotalFlights INT,
                   TotalCancelledFlights INT,
                   AvgArrDelay Float,
                   AvgDepDelay Float,
                   AvgDistance Float,
                   OnTimeFlights int,
                   OTP_Pct FLOAT,
                   CancelRate_PCT FLOAT
                )
""")

#create carrier_kpis table
cursor.execute("""
               create table if not exists carrier_kpis(
                   Reporting_Airline VARCHAR(10),
                   TotalFlights INT,
                   AvgArrDelay Float,
                   Total_On_Time_Flights INT,
                   Cancelled_flights INT,
                   Total_Weather_Delay Float,
                   Total_Carrier_Delay Float,
                   Total_NAS_Delay Float,
                   OTP_pct Float
                   )
""")

#create day_of_week_heatmap table
cursor.execute("""
               Create Table if not exists dow_heatmap(
                   DayName Varchar(20),
                   DayOfWeek Int,
                   Origin Varchar(10),
                   AvgArrDelay Float,
                   Total_Flights Int
                   )
               
""")

connection.commit()
print("All tables created in MYSQL!")
connection.close()

All tables created in MYSQL!


In [0]:
#Push Gold Tables to MySQL

from sqlalchemy import create_engine

#Create SQLAlchemy engine
engine= create_engine(
    f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}"
)

#Convert route_kpis spark dataframe to pandas then load to mysql
print("Loading route_kpis....")
route_kpis_path = "/Volumes/workspace/default/airlines/gold/route_kpis"
route_kpis_df= spark.read.format("delta").load(route_kpis_path)
route_kpis_df.toPandas().to_sql("route_kpis",engine, if_exists="append",index=False)
print("*** route_kpis loaded!")

#Convert carrier_kpis spark dataframe to pandas then load to mysql
print("Loading carrier_kpis....")
carrier_kpis_path = "/Volumes/workspace/default/airlines/gold/carrier_kpis"
carrier_kpis_df= spark.read.format("delta").load(carrier_kpis_path)
carrier_kpis_df.toPandas().to_sql("carrier_kpis", engine, if_exists="append",index=False)
print("*** carrier_kpis loaded!")

#Convert day_of_week_heatmap spark dataframe to pandas then load to mysql
day_of_week_heatmap_path= "/Volumes/workspace/default/airlines/gold/day_of_week_heatmap"
dow_heatmap_df= spark.read.format("delta").load(day_of_week_heatmap_path)
dow_heatmap_df.toPandas().to_sql("dow_heatmap",engine, if_exists="append", index=False)
print("*** dow_heatmap loaded!")

print("\n ALL DATA LOADED TO MYSQL!")

Loading route_kpis....
*** route_kpis loaded!
Loading carrier_kpis....
*** carrier_kpis loaded!
*** dow_heatmap loaded!

 ALL DATA LOADED TO MYSQL!


In [0]:
print(route_kpis_df.columns)

['Origin', 'Dest', 'Route', 'Month', 'TotalFlights', 'TotalCancelledFlights', 'AvgArrDelay', 'AvgDepDelay', 'AvgDistance', 'OnTimeFlights', 'OTP_Pct', 'CancelRate_PCT']
